# Week 4: Gold aggregates

Decisions (logged in gold_build_log.md)

1. Headline measure: payment index = Σ actual total payment ÷ Σ (discharges × DRG median payment), so it adjusts for case mix. 1.00 = paid the national median for that mix of DRGs. Raw payment per discharge and weighted markup are kept as descriptive columns.

2. Thin DRGs: all 539 benchmarks in the expected value. Non-thin only (310) is run as a sensitivity check.

3. Hospital outliers: 3 × IQR above Q3 on the hospital index (same rule as DQ-14). Hospitals with fewer than 5 DRGs are not flagged (DQ-17).

4. RUCA: binary urban_rural is the headline. The four RUCA groups are kept in the table. Unknown (99) is its own bucket (DQ-12).

5. Region: gold_dim_state carries Census region (4) and division (9). Regions are used for reporting.

6. Maryland: kept in state summaries with markup. Payment index is null because Maryland has no benchmark (DQ-13).

In [0]:
--- State dimension: one row per state (50 states + DC) with its Census region and division. 

--- Source: US Census Bureau regions and divisions. DC is in South / South Atlantic. 

--- Checks: 51 rows, 51 distinct states, every hospital in gold_dim_hospital matches. 

CREATE OR REPLACE TABLE workspace.cms.gold_dim_state AS
SELECT * 
FROM VALUES
     -- Northeast / New England (6)
    ('CT', 'Northeast', 'New England'),
    ('ME', 'Northeast', 'New England'),
    ('MA', 'Northeast', 'New England'),
    ('NH', 'Northeast', 'New England'),
    ('RI', 'Northeast', 'New England'),
    ('VT', 'Northeast', 'New England'),

      -- Northeast / Middle Atlantic (3)
    ('NJ', 'Northeast', 'Middle Atlantic'),
    ('NY', 'Northeast', 'Middle Atlantic'),
    ('PA', 'Northeast', 'Middle Atlantic'),

    -- Midwest / East North Central (5)
    ('IL', 'Midwest', 'East North Central'),
    ('IN', 'Midwest', 'East North Central'),
    ('MI', 'Midwest', 'East North Central'),
    ('OH', 'Midwest', 'East North Central'),
    ('WI', 'Midwest', 'East North Central'),

    -- Midwest / West North Central (7)
    ('IA', 'Midwest', 'West North Central'),
    ('KS', 'Midwest', 'West North Central'),
    ('MN', 'Midwest', 'West North Central'),
    ('MO', 'Midwest', 'West North Central'),
    ('NE', 'Midwest', 'West North Central'),
    ('ND', 'Midwest', 'West North Central'),
    ('SD', 'Midwest', 'West North Central'),

    -- South / South Atlantic (9, includes DC)
    ('DE', 'South', 'South Atlantic'),
    ('DC', 'South', 'South Atlantic'),
    ('FL', 'South', 'South Atlantic'),
    ('GA', 'South', 'South Atlantic'),
    ('MD', 'South', 'South Atlantic'),
    ('NC', 'South', 'South Atlantic'),
    ('SC', 'South', 'South Atlantic'),
    ('VA', 'South', 'South Atlantic'),
    ('WV', 'South', 'South Atlantic'),

    -- South / East South Central (4)
    ('AL', 'South', 'East South Central'),
    ('KY', 'South', 'East South Central'),
    ('MS', 'South', 'East South Central'),
    ('TN', 'South', 'East South Central'),

    -- South / West South Central (4)
    ('AR', 'South', 'West South Central'),
    ('LA', 'South', 'West South Central'),
    ('OK', 'South', 'West South Central'),
    ('TX', 'South', 'West South Central'),

    -- West / Mountain (8)
    ('AZ', 'West', 'Mountain'),
    ('CO', 'West', 'Mountain'),
    ('ID', 'West', 'Mountain'),
    ('MT', 'West', 'Mountain'),
    ('NV', 'West', 'Mountain'),
    ('NM', 'West', 'Mountain'),
    ('UT', 'West', 'Mountain'),
    ('WY', 'West', 'Mountain'),

    -- West / Pacific (5)
    ('AK', 'West', 'Pacific'),
    ('CA', 'West', 'Pacific'),
    ('HI', 'West', 'Pacific'),
    ('OR', 'West', 'Pacific'),
    ('WA', 'West', 'Pacific')
AS t(state_abbr, census_region, census_division); 




num_affected_rows,num_inserted_rows


In [0]:
--- Check 1 and 2: expect 51 and 51
SELECT COUNT (*) AS n_rows,
    COUNT(DISTINCT state_abbr) AS n_states
FROM workspace.cms.gold_dim_state;

n_rows,n_states
51,51


In [0]:
--- Check 3: hospitals with no region; expect 0
SELECT COUNT(*) AS unmatched_hospitals
FROM workspace.cms.gold_dim_hospital h 
LEFT JOIN workspace.cms.gold_dim_state s 
    ON h.state_abbr = s.state_abbr
WHERE s.state_abbr IS NULL;

unmatched_hospitals
0


In [0]:
--- Check 4: hospitals per region; should sum to 2,906
SELECT s.census_region,
    COUNT(*) AS n_hospitals
FROM workspace.cms.gold_dim_hospital h 
JOIN workspace.cms.gold_dim_state s 
    ON h.state_abbr = s.state_abbr
GROUP BY s.census_region
ORDER BY s.census_region;

census_region,n_hospitals
Midwest,685
Northeast,437
South,1203
West,581


In [0]:
--- Check 5: joined row count; expect 2,906 (a duplicate state row would inflate it)
SELECT COUNT(*) AS n_joined
FROM workspace.cms.gold_dim_hospital h
JOIN workspace.cms.gold_dim_state s
  ON h.state_abbr = s.state_abbr; 

n_joined
2906


In [0]:
--- State summary: one row per state with volume, payment, markup and the case-mix-adjusted payment index. 

--- Payment index = actual total payment / expected payment, where expected = discharges x DRG median (gold_drg_benchmark).

--- All ratios are totals over totals (discharge-weighted, DQ-04). Maryland index is NULL (no benchmark, DQ-13).

--- Checks: 51 rows; NV markup 9.10, MA 2.67; Maryland index NULL; payments sum to 90,927,479,550. 

CREATE OR REPLACE TABLE workspace.cms.gold_state_summary AS 
SELECT
    h.state_abbr,
    s.census_region,
    s.census_division,
    COUNT(DISTINCT f.ccn) < 10 AS is_small_state,
    COUNT(DISTINCT f.ccn) AS n_hospitals,
    SUM(f.tot_discharges) AS tot_discharges,
    SUM(f.tot_total_payment) AS tot_payment,
    SUM(f.tot_submitted_charge) AS tot_charges,
    SUM(f.tot_total_payment) / SUM(f.tot_discharges) AS payment_per_discharge,
    SUM(f.tot_submitted_charge) / SUM(f.tot_total_payment) AS markup_ratio, 
    SUM(f.tot_discharges * b.median_payment) AS expected_payment,
    CASE WHEN MAX(CAST(h.is_maryland AS INT)) = 1 THEN null
        ELSE SUM(f.tot_total_payment) / SUM(f.tot_discharges * b.median_payment)
    END AS payment_index

FROM workspace.cms.gold_fact_inpatient f 
JOIN workspace.cms.gold_dim_hospital h ON f.ccn = h.ccn
JOIN workspace.cms.gold_dim_state s ON h.state_abbr = s.state_abbr
LEFT JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd 
GROUP BY h.state_abbr, s.census_region, s.census_division;
   


num_affected_rows,num_inserted_rows


In [0]:
--- Check: one row per state.
--- Expected: 51 (50 states + DC), matching gold_dim_state. 

SELECT COUNT(*) AS n_states 
FROM workspace.cms.gold_state_summary

n_states
51


In [0]:
--- Check: spot-check markup against the Silver results, and confirm the Maryland rule. 
--- Expected: NV markup 9.10, MA 2.67 (highest and lowest, from silver_cleaning_log.md).
--- MD payment_index must be NULL (no IPPS benchmark, DQ-13).

SELECT 
    state_abbr,
    ROUND(markup_ratio, 2) AS markup,
    ROUND(payment_index, 3) AS payment_index 

FROM workspace.cms.gold_state_summary
WHERE state_abbr IN ('NV', 'MA', 'MD');

state_abbr,markup,payment_index
NV,9.10,1.1
MA,2.67,1.226
MD,1.11,null


In [0]:
--- Check: reconciliation. Total payment across states must equal the Silver total, 
--- which proves the joins neither dropped nor duplicated rows.
--- Expected: 90,927,479,550.00

SELECT ROUND(SUM(tot_payment), 2) AS total_payment FROM workspace.cms.gold_state_summary;

total_payment
90927479550.00


In [0]:
--- Check: fact rows with no DRG benchmark, outside Maryland.
--- A missing benchmark would understate expected payment and inflate that state's index. 
--- Expected: 0 (the only unbenchmarked DRG, 950, is billed only in Maryland, DQ-22).

SELECT COUNT(*) AS missing_benchmark_rows
FROM workspace.cms.gold_fact_inpatient f  
JOIN workspace.cms.gold_dim_hospital h ON f.ccn = h.ccn
LEFT JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd 
WHERE b.drg_cd IS NULL AND h.is_maryland = false;

missing_benchmark_rows
0


In [0]:
--- Result: states ranked by case-mix-adjusted payment index (highest first).
--- payment_index 1.00 = paid the national DRG median for that state's mix of cases. 
--- Maryland sorts last (NULL index). Rounded for display only (DQ-05).

SELECT state_abbr, census_region, n_hospitals,
    ROUND(payment_per_discharge, 0) AS pay_per_dschg,
    ROUND(markup_ratio, 2) AS markup,
    ROUND(payment_index, 3) AS payment_index

FROM workspace.cms.gold_state_summary
ORDER BY payment_index DESC;

state_abbr,census_region,n_hospitals,pay_per_dschg,markup,payment_index
NY,Northeast,125,24595,4.97,1.449
AK,West,8,24317,5.59,1.427
CA,West,272,24038,6.34,1.427
HI,West,11,23306,3.39,1.414
DC,South,6,32967,4.28,1.383
CT,Northeast,25,20929,3.38,1.3
WY,West,9,17612,3.00,1.262
MA,Northeast,54,19834,2.67,1.226
OR,West,31,21903,3.19,1.21
RI,Northeast,9,17737,2.74,1.184


In [0]:
--- Region summary: Census division, region and national rows in one table via ROLLUP.
--- geo_level tells the grain apart: 'division', 'region', or 'national'.
--- Volume and markup include Maryland; the payment index excludes Maryland from both actual and expected payment (no IPPS benchmark, DQ-13). Totals over totals (DQ-04).
--- Checks: 14 rows (9+4+1); national markup 5.03 and payment per discharge 18,359.99

CREATE OR REPLACE TABLE workspace.cms.gold_region_summary AS 
SELECT
    CASE WHEN GROUPING(s.census_region) = 1 THEN 'national'
         WHEN GROUPING(s.census_division) = 1 THEN 'region'
         ELSE 'division' END AS geo_level,
         
    s.census_region, 
    s.census_division,
    COUNT(DISTINCT f.ccn) AS n_hospitals,
    SUM(f.tot_discharges) AS tot_discharges,
    SUM(f.tot_total_payment) AS tot_payment,
    SUM(f.tot_total_payment) / SUM(f.tot_discharges) AS payment_per_discharge,
    SUM(f.tot_submitted_charge) / SUM(f.tot_total_payment) AS markup_ratio,
    SUM(CASE WHEN NOT h.is_maryland THEN f.tot_total_payment END) / SUM(CASE WHEN NOT h.is_maryland THEN f.tot_discharges * b.median_payment END) AS payment_index

FROM workspace.cms.gold_fact_inpatient f 
JOIN workspace.cms.gold_dim_hospital h ON f.ccn = h.ccn
JOIN workspace.cms.gold_dim_state s ON h.state_abbr = s.state_abbr
LEFT JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd 
GROUP BY ROLLUP (s.census_region, s.census_division); 

num_affected_rows,num_inserted_rows


In [0]:
--- Check: rows per grain from the ROLLUP.
--- Expected: 9 division, 4 region, 1 national.

SELECT geo_level, COUNT(*) AS n_rows
FROM workspace.cms.gold_region_summary
GROUP BY geo_level
ORDER BY geo_level;

geo_level,n_rows
division,9
national,1
region,4


In [0]:
--- Check: national row reconciles to the Silver totals.
--- Expected: markup 5.03, payment per discharge 18,359.99, total payment 90,927,479,550.00

SELECT 
    ROUND(markup_ratio, 2) AS markup,
    ROUND(payment_per_discharge, 2) AS pay_per_dschg,
    ROUND(tot_payment, 2) AS total_payment,
    ROUND(payment_index, 3) AS payment_index

FROM workspace.cms.gold_region_summary
WHERE geo_level = 'national';

markup,pay_per_dschg,total_payment,payment_index
5.03,18359.99,90927479550.00,1.104


In [0]:
--- Check: the four regions add up to the national total.
--- Expected: 90,927,479,550.00 (same as cell 15)

SELECT ROUND(SUM(tot_payment), 2) AS region_total
FROM workspace.cms.gold_region_summary
WHERE geo_level = 'region';

region_total
90927479550.00


In [0]:
--- Result: regions, then divisions, ranked by payment index within each level.
--- The nationabl index (cell 15) is the reference line. Rounded for display only (DQ-05).

SELECT geo_level, census_region, census_division, n_hospitals,
    ROUND(payment_per_discharge, 0) AS pay_per_dschg,
    ROUND(markup_ratio, 2) AS markup,
    ROUND(payment_index, 3) AS payment_index

FROM workspace.cms.gold_region_summary
WHERE geo_level <> 'national'
ORDER BY geo_level DESC, payment_index DESC;

geo_level,census_region,census_division,n_hospitals,pay_per_dschg,markup,payment_index
region,West,null,581,21687,5.86,1.271
region,Northeast,null,437,20409,4.81,1.252
region,Midwest,null,685,17004,4.13,1.026
region,South,null,1203,16575,5.18,0.986
division,West,Pacific,367,23311,5.87,1.373
division,Northeast,Middle Atlantic,313,20798,5.51,1.268
division,Northeast,New England,124,19463,2.98,1.214
division,West,Mountain,214,18002,5.85,1.044
division,Midwest,East North Central,459,16675,4.11,1.036
division,South,South Atlantic,531,17136,4.81,1.017


In [0]:
--- Preview of the base CTE for gold_hospital_index (not saved as a table yet).
--- Grain: one row per non-Maryland hospital (Maryland has no IPPS benchmark, DQ-13).
--- payment_index = actual total payment / expected payment (discharges x DRG median), totals over totals (DQ-04).
--- Look for: index mostly between 0.6 and 2.0, n_drgs from 1 to a few hundred, no NULL expected_payment.

SELECT f.ccn, h.state_abbr, h.provider_city, h.provider_name,
    COUNT(*) AS n_drgs,
    SUM(f.tot_discharges) AS tot_discharges,
    SUM(f.tot_total_payment) AS tot_payment,
    SUM(f.tot_discharges * b.median_payment) AS expected_payment,
    SUM(f.tot_total_payment) / SUM(f.tot_discharges * b.median_payment) AS payment_index

FROM workspace.cms.gold_fact_inpatient f  
JOIN workspace.cms.gold_dim_hospital h on f.ccn = h.ccn
JOIN workspace.cms.gold_drg_benchmark b on f.drg_cd = b.drg_cd
WHERE NOT h.is_maryland
GROUP BY f.ccn, h.state_abbr, h.provider_city, h.provider_name
ORDER BY payment_index DESC
LIMIT 10;

ccn,state_abbr,provider_city,provider_name,n_drgs,tot_discharges,tot_payment,expected_payment,payment_index
450289,TX,Houston,Harris Health,12,254,32728601.9998000,3289524.51675035,9.949341259852313
140300,IL,Chicago,Provident Hospital Of Chicago,1,16,1387558.0000000,174871.126864,7.934746146395715
050717,CA,Downey,Lac/Rancho Los Amigos National Rehabilitation Ctr,2,23,1165216.0000040,201241.44772875,5.7901392240756255
140124,IL,Chicago,John H Stroger Jr Hospital,16,318,22862369.9999090,4421223.51112625,5.171050489163147
330202,NY,Brooklyn,Kings County Hospital Center,34,902,52914746.0001080,1.2693918447424002E7,4.168511576568863
330199,NY,New York,Metropolitan Hospital Center,2,45,2523033.0000000,621434.43838,4.060014772559474
330396,NY,Brooklyn,Woodhull Medical & Mental Health Center,9,196,10086772.0000030,2570941.6267965,3.923376514997557
290012,NV,Henderson,Saint Rose Dominican Hospitals - Rose De Lima,1,14,433121.9999960,113366.9999994,3.8205297837844556
450015,TX,Dallas,Parkland Health And Hospital System,44,1119,65278924.9999170,1.7351698480529398E7,3.7621057715570303
330231,NY,Jamaica,Queens Hospital Center,21,570,32138264.9999790,8604585.48024295,3.7350160648379744


In [0]:
--- Base CTE plus the Week 3 outlier count per hospital.
--- n_outlier_cells = number of this hospital's DRG cells flagged by DQ-14 (is_payment_outlier).
--- The deviation table has the same grain as the fact table (hospital x DRG), so it joins on
--- BOTH ccn and drg_cd; joining on ccn alone would fan out and multiply every sum by n_drgs.
--- Check: 2,862 rows (2,906 hospitals minus 44 in Maryland).

SELECT f.ccn, h.state_abbr, h.provider_city, h.provider_name, h.urban_rural, s.census_region,
    COUNT(*) AS n_drgs,
    COUNT(*) <5 AS is_low_drg_hospital,
    SUM(f.tot_discharges) AS tot_discharges,
    SUM(f.tot_total_payment) AS tot_payment,
    SUM(f.tot_discharges * b.median_payment) AS expected_payment,
    SUM(f.tot_total_payment) / SUM(f.tot_discharges * b.median_payment) AS payment_index,
    SUM(CASE WHEN d.is_payment_outlier THEN 1 ELSE 0 END) AS n_outlier_cells,
    SUM(CASE WHEN d.is_payment_outlier THEN 1 ELSE 0 END) / COUNT(*) AS share_outlier_cells

FROM workspace.cms.gold_fact_inpatient f 
JOIN workspace.cms.gold_dim_hospital h ON f.ccn = h.ccn
JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd
LEFT JOIN workspace.cms.gold_hospital_drg_deviation d ON f.ccn = d.ccn AND f.drg_cd = d.drg_cd
JOIN workspace.cms.gold_dim_state s ON h.state_abbr = s.state_abbr
WHERE NOT h.is_maryland
GROUP BY f.ccn, h.state_abbr, h.provider_city, h.provider_name, h.urban_rural, s.census_region

ccn,state_abbr,provider_city,provider_name,urban_rural,census_region,n_drgs,is_low_drg_hospital,tot_discharges,tot_payment,expected_payment,payment_index,n_outlier_cells,share_outlier_cells
010065,AL,Alexander City,Russell Medical Center,Rural,South,4,true,107,1256615.9999796,1474330.1768128998,0.8523301087793385,0,0.0
010101,AL,Talladega,Citizens Baptist Medical Center,Unknown,South,5,false,116,1447600.0000050,1580832.07253,0.9157202875370739,0,0.0
010125,AL,Haleyville,Lakeland Community Hospital,Rural,South,9,false,232,2056105.9999992,2191514.9652905003,0.9382121649014836,0,0.0
010157,AL,Muscle Shoals,North Alabama Shoals Hospital,Urban,South,1,true,21,170853.0000009,171363.0833334,0.9970233767823401,0,0.0
030061,AZ,Sun City,Banner Boswell Medical Center,Urban,West,113,false,4057,56729426.9998868,6.411291782773206E7,0.8848361441342555,0,0.0
040016,AR,Little Rock,Uams Medical Center,Urban,South,115,false,3092,86590388.9999243,8.027362911022851E7,1.0786903489939625,0,0.0
040027,AR,Mountain Home,Baxter Health,Rural,South,68,false,1924,21139411.9999241,2.833547525410615E7,0.7460404955396238,0,0.0
040161,AR,Bryant,Arkansas Heart Hospital-Encore,Urban,South,15,false,353,4169199.0000025,5876326.726650099,0.7094906723097106,0,0.0
050047,CA,San Francisco,California Pacific Medical Center- Van Ness Campus,Urban,West,96,false,2595,120190292.9997514,6.736573969091307E7,1.7841456733230796,13,0.13541666666666666
050104,CA,Lynwood,Saint Francis Medical Center,Urban,West,39,false,1248,25920968.0000143,1.9919409958456904E7,1.3012919586510845,0,0.0


In [0]:
--- Hospital index: one row per non-Maryland hospital (2,862) with the case-mix-adjusted payment index.
--- payment_index = actual total payment / expected payment (discharges x DRG median), totals over totals (DQ-04).
--- Maryland excluded: no IPPS benchmark (DQ-13).
--- base: the validated cell 19 query (volume, index, Week 3 outlier-cell count and share, low-DRG flag).
--- thresholds: Q1 and Q3 of payment_index across hospitals with at least 5 DRGs (DQ-17).
--- is_hospital_outlier: index above Q3 + 3 x IQR, upper fence only, never set for low-DRG hospitals (DQ-14, DQ-17).
--- Checks: 2,862 rows; index across all rows = national 1.104; low-DRG hospitals never flagged.

CREATE OR REPLACE TABLE workspace.cms.gold_hospital_index AS
WITH base AS (
    SELECT f.ccn, h.state_abbr, h.provider_city, h.provider_name, h.urban_rural, s.census_region,
        COUNT(*) AS n_drgs,
        COUNT(*) <5 AS is_low_drg_hospital,
        SUM(f.tot_discharges) AS tot_discharges,
        SUM(f.tot_total_payment) AS tot_payment,
        SUM(f.tot_discharges * b.median_payment) AS expected_payment,
        SUM(f.tot_total_payment) / SUM(f.tot_discharges * b.median_payment) AS payment_index,
        SUM(CASE WHEN d.is_payment_outlier THEN 1 ELSE 0 END) AS n_outlier_cells,
        SUM(CASE WHEN d.is_payment_outlier THEN 1 ELSE 0 END) / COUNT(*) AS share_outlier_cells

        FROM workspace.cms.gold_fact_inpatient f 
JOIN workspace.cms.gold_dim_hospital h ON f.ccn = h.ccn
JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd
LEFT JOIN workspace.cms.gold_hospital_drg_deviation d ON f.ccn = d.ccn AND f.drg_cd = d.drg_cd
JOIN workspace.cms.gold_dim_state s ON h.state_abbr = s.state_abbr
WHERE NOT h.is_maryland
GROUP BY f.ccn, h.state_abbr, h.provider_city, h.provider_name, h.urban_rural, s.census_region
), 

thresholds AS (
    SELECT PERCENTILE(payment_index, 0.25) AS q1, PERCENTILE(payment_index, 0.75) AS q3
    FROM base
    WHERE n_drgs >= 5    
)

SELECT *, 
    payment_index > q3 + 3 * (q3 - q1) AND n_drgs >= 5 AS is_hospital_outlier
FROM base
CROSS JOIN thresholds;

num_affected_rows,num_inserted_rows


In [0]:
--- Check: one row per non-Maryland hospital.
--- Expected: 2,862 (2,906 hospitals minus 44 in Maryland).

SELECT * FROM workspace.cms.gold_hospital_index;

ccn,state_abbr,provider_city,provider_name,urban_rural,census_region,n_drgs,is_low_drg_hospital,tot_discharges,tot_payment,expected_payment,payment_index,n_outlier_cells,share_outlier_cells,q1,q3,is_hospital_outlier
010007,AL,Opp,Mizell Memorial Hospital,Rural,South,5,false,80,717422.0000032,787193.8751579,0.9113663388949708,0,0.0,0.8894117739800582,1.1397766839045782,false
010049,AL,Enterprise,Medical Center Enterprise,Rural,South,12,false,255,2470017.9999731,3096909.2328884,0.7975751997320838,0,0.0,0.8894117739800582,1.1397766839045782,false
010052,AL,Dadeville,Lake Martin Community Hospital,Urban,South,1,true,18,133898.9999994,130300.7142852,1.0276152416657067,0,0.0,0.8894117739800582,1.1397766839045782,false
010079,AL,Athens,Athens Limestone Hospital,Rural,South,26,false,569,6292817.9999781,7638783.225070499,0.8237984787060145,0,0.0,0.8894117739800582,1.1397766839045782,false
010104,AL,Birmingham,Grandview Medical Center,Urban,South,120,false,4006,53739283.0000830,6.5272739321916446E7,0.8233036265729253,0,0.0,0.8894117739800582,1.1397766839045782,false
010139,AL,Vestavia,Brookwood Baptist Medical Center,Urban,South,41,false,906,13887110.9999773,1.67362968024387E7,0.8297600815703605,0,0.0,0.8894117739800582,1.1397766839045782,false
020001,AK,Anchorage,Providence Alaska Medical Center,Urban,West,129,false,3995,96851580.9996885,7.604702172112837E7,1.2735749383434425,3,0.023255813953488372,0.8894117739800582,1.1397766839045782,false
030089,AZ,Glendale,Banner Thunderbird Medical Center,Urban,West,76,false,2368,36708355.9999712,3.6654720149886146E7,1.0014632726662687,0,0.0,0.8894117739800582,1.1397766839045782,false
030121,AZ,Mesa,Honorhealth Mountain Vista Medical Center,Urban,West,23,false,436,6904158.9999549,6580191.257280549,1.049233788199682,0,0.0,0.8894117739800582,1.1397766839045782,false
040161,AR,Bryant,Arkansas Heart Hospital-Encore,Urban,South,15,false,353,4169199.0000025,5876326.726650099,0.7094906723097106,0,0.0,0.8894117739800582,1.1397766839045782,false


In [0]:
--- Check: reconciliation. Total actual payment over total expected payment across all hospitals
--- must equal the national index from the region summary, which proves no rows were dropped or duplicated.
--- Expected: 1.104

SELECT SUM(tot_payment) / SUM(expected_payment) FROM workspace.cms.gold_hospital_index;

SUM(tot_payment)/SUM(expected_payment)
1.1041924255865647


In [0]:
--- Check: low-DRG hospitals are never flagged as outliers (DQ-17).
--- Expected: 0

SELECT COUNT (*) AS n_low_drg_flagged
FROM workspace.cms.gold_hospital_index 
WHERE is_low_drg_hospital AND is_hospital_outlier; 

n_low_drg_flagged
0


In [0]:
--- Check: the outlier thresholds. Every row carries the same q1 and q3, so one row is enough.
--- Upper fence = Q3 + 3 x IQR across hospitals with at least 5 DRGs (DQ-14).
--- Expected: Q1 about 0.889, Q3 about 1.140, fence about 1.891.

SELECT q1, q3, q3 + 3 * (q3 - q1) AS fence
FROM workspace.cms.gold_hospital_index
LIMIT 1;

q1,q3,fence
0.8894117739800582,1.1397766839045782,1.8908714136781382


In [0]:
--- Check: number of hospitals flagged as payment outliers.
--- Small share of the 2,493 hospitals with at least 5 DRGs expected; recorded in the build log.

SELECT COUNT(*) AS n_hospitals_outliers
FROM workspace.cms.gold_hospital_index
WHERE is_hospital_outlier;

n_hospitals_outliers
51


In [0]:
--- Result: hospital payment outliers, highest index first.
--- share_outlier_cells shows whether a hospital is high across most DRGs or only a few.
--- High index can reflect teaching, safety-net and wage-index add-ons, not only inefficiency (DQ-23).

SELECT 
    provider_name, 
    provider_city, 
    state_abbr, 
    census_region, 
    urban_rural, 
    n_drgs, 
    tot_discharges, 
    tot_payment, 
    expected_payment, 
    ROUND (payment_index, 3) AS payment_index,
    ROUND (share_outlier_cells, 2) AS share_outlier_cells,
    n_outlier_cells  

FROM workspace.cms.gold_hospital_index
WHERE is_hospital_outlier
ORDER BY payment_index DESC;

provider_name,provider_city,state_abbr,census_region,urban_rural,n_drgs,tot_discharges,tot_payment,expected_payment,payment_index,share_outlier_cells,n_outlier_cells
Harris Health,Houston,TX,South,Urban,12,254,32728601.9998000,3289524.51675035,9.949,1.0,12
John H Stroger Jr Hospital,Chicago,IL,Midwest,Urban,16,318,22862369.9999090,4421223.51112625,5.171,1.0,16
Kings County Hospital Center,Brooklyn,NY,Northeast,Urban,34,902,52914746.0001080,1.2693918447424002E7,4.169,1.0,34
Woodhull Medical & Mental Health Center,Brooklyn,NY,Northeast,Urban,9,196,10086772.0000030,2570941.6267965,3.923,1.0,9
Parkland Health And Hospital System,Dallas,TX,South,Urban,44,1119,65278924.9999170,1.7351698480529398E7,3.762,0.89,39
Queens Hospital Center,Jamaica,NY,Northeast,Urban,21,570,32138264.9999790,8604585.48024295,3.735,0.95,20
Kaiser Foundation Hospital - San Francisco,San Francisco,CA,West,Urban,7,115,7446143.9999680,2239268.7852632497,3.325,1.0,7
Kaiser Foundation Hospital-Santa Clara,Santa Clara,CA,West,Urban,8,235,10665615.0000030,3485208.5291746,3.06,0.5,4
Contra Costa Regional Medical Center,Martinez,CA,West,Urban,13,305,9562714.9999880,3196555.2648698497,2.992,1.0,13
Memorial Hospital Los Banos,Los Banos,CA,West,Rural,8,172,6384653.0000260,2145292.9259139,2.976,1.0,8


In [0]:
--- RUCA summary: payment by rural/urban status, Maryland excluded (no IPPS benchmark, DQ-13).
--- fact_ruca CTE: one row per non-Maryland hospital x DRG with ruca_group (ruca_code cast to DOUBLE, floored, mapped by CASE).
--- ELSE NULL on purpose: an unmapped code shows up as a NULL group instead of hiding in Unknown.
--- ROLLUP(urban_rural, ruca_group) gives 5 ruca_group rows, 3 urban_rural rows and 1 national row, labeled by ruca_level.
--- Headline grain is urban_rural; Small town and Rural groups are thin (W4-4). Unknown kept, never imputed (DQ-12).
--- Ratios are totals over totals (DQ-04). Checks: 9 rows; hospital counts and markups match Silver; national index 1.104.

CREATE OR REPLACE TABLE workspace.cms.gold_ruca_summary AS
WITH fact_ruca AS (
    SELECT
        f.ccn,
        f.tot_discharges,
        f.tot_total_payment,
        f.tot_submitted_charge,
        b.median_payment,
        h.urban_rural,
        CASE
            WHEN FLOOR(CAST(h.ruca_code AS DOUBLE)) BETWEEN 1 AND 3 THEN 'Metropolitan'
            WHEN FLOOR(CAST(h.ruca_code AS DOUBLE)) BETWEEN 4 AND 6 THEN 'Micropolitan'
            WHEN FLOOR(CAST(h.ruca_code AS DOUBLE)) BETWEEN 7 AND 9 THEN 'Small town'
            WHEN FLOOR(CAST(h.ruca_code AS DOUBLE)) = 10 THEN 'Rural'
            WHEN FLOOR(CAST(h.ruca_code AS DOUBLE)) = 99 THEN 'Unknown'
            ELSE NULL
        END AS ruca_group
    FROM workspace.cms.gold_fact_inpatient f
    JOIN workspace.cms.gold_dim_hospital h  ON f.ccn = h.ccn
    JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd
    WHERE NOT h.is_maryland
)
SELECT
    CASE WHEN GROUPING(urban_rural) = 1 THEN 'national'
         WHEN GROUPING(ruca_group) = 1  THEN 'urban_rural'
         ELSE 'ruca_group' END AS ruca_level,
    urban_rural,
    ruca_group,
    COUNT(DISTINCT ccn) AS n_hospitals,
    SUM(tot_discharges) AS tot_discharges,
    SUM(tot_total_payment) AS tot_payment,
    SUM(tot_total_payment) / SUM(tot_discharges) AS payment_per_discharge,
    SUM(tot_submitted_charge) / SUM(tot_total_payment) AS markup_ratio,
    SUM(tot_discharges * median_payment) AS expected_payment,
    SUM(tot_total_payment) / SUM(tot_discharges * median_payment) AS payment_index
FROM fact_ruca
GROUP BY ROLLUP(urban_rural, ruca_group);

num_affected_rows,num_inserted_rows


In [0]:
--- Check: rows per grain from the ROLLUP, and no unmapped RUCA codes.
--- Expected: ruca_group 5, urban_rural 3, national 1 (9 total).
--- A ruca_group row with a NULL ruca_group would mean a code fell through the CASE (ELSE NULL).

SELECT ruca_level, COUNT(*) AS n_rows
FROM workspace.cms.gold_ruca_summary
GROUP BY ruca_level
ORDER BY ruca_level;

ruca_level,n_rows
national,1
ruca_group,5
urban_rural,3


In [0]:
--- Check: RUCA group rows against the Silver log (Maryland excluded).
--- Expected hospitals: Metropolitan 2,101 · Micropolitan 515 · Small town 196 · Rural 38 · Unknown 12.
--- Expected markup: 5.25 · 3.88 · 3.32 · 4.08 · 5.59.
--- Expected payment per discharge: 18,651.20 · 13,221.78 · 13,222.17 · 12,685.63 · 16,119.50.

SELECT 
    ruca_group, 
    n_hospitals, 
    ROUND (markup_ratio, 2) AS markup_ratio,
    ROUND (payment_per_discharge, 2) AS payment_per_discharge
FROM workspace.cms.gold_ruca_summary
WHERE ruca_level = 'ruca_group'
ORDER BY ruca_group;

ruca_group,n_hospitals,markup_ratio,payment_per_discharge
Metropolitan,2101,5.25,18651.20
Micropolitan,515,3.88,13221.78
Rural,38,4.08,12685.63
Small town,196,3.32,13222.17
Unknown,12,5.59,16119.50


In [0]:
--- Check: national row against the Silver log (all states + DC, Maryland excluded).
--- Expected: 2,862 hospitals, 4,821,831 discharges, payment per discharge 18,215.26, markup 5.17,
--- payment index 1.104 (same as the hospital index and the region summary).

SELECT
    n_hospitals, 
    tot_discharges,
    ROUND (payment_per_discharge, 2) AS payment_per_discharge,
    ROUND (markup_ratio, 2) AS markup_ratio,
    ROUND (payment_index, 3) AS payment_index
FROM workspace.cms.gold_ruca_summary 
WHERE ruca_level = 'national';

n_hospitals,tot_discharges,payment_per_discharge,markup_ratio,payment_index
2862,4821831,18215.26,5.17,1.104


In [0]:
--- Result: payment index by urban/rural (headline) and RUCA group (detail), Maryland excluded.
--- 1.00 = paid the national DRG median for that group's mix of cases; national average is 1.104.
--- Small town and Rural groups are thin (W4-4). Rounded for display only (DQ-05).

SELECT 
    ruca_group,
    urban_rural,
    ruca_level,
    n_hospitals, 
    tot_discharges,
    ROUND (payment_per_discharge, 2) AS payment_per_discharge,
    ROUND (markup_ratio, 2) AS markup_ratio,
    ROUND (payment_index, 3) AS payment_index

FROM workspace.cms.gold_ruca_summary
WHERE ruca_level <> 'national'
ORDER BY ruca_level DESC, ruca_group DESC;

ruca_group,urban_rural,ruca_level,n_hospitals,tot_discharges,payment_per_discharge,markup_ratio,payment_index
null,Urban,urban_rural,2101,4425789,18651.20,5.25,1.111
null,Rural,urban_rural,749,376491,13199.46,3.83,1.001
null,Unknown,urban_rural,12,19551,16119.50,5.59,1.022
Unknown,Unknown,ruca_group,12,19551,16119.50,5.59,1.022
Small town,Rural,ruca_group,196,37260,13222.17,3.32,1.048
Rural,Rural,ruca_group,38,15699,12685.63,4.08,0.987
Micropolitan,Rural,ruca_group,515,323532,13221.78,3.88,0.996
Metropolitan,Urban,ruca_group,2101,4425789,18651.20,5.25,1.111


In [0]:
--- Sensitivity check (W4-2): state payment index using only non-thin DRG benchmarks (310 DRGs with 30+ hospitals).
--- Thin DRGs are dropped from BOTH actual and expected payment, so the ratio stays like-for-like. Maryland excluded (DQ-13).
--- Compared with the main index in gold_state_summary (all 539 benchmarks).
--- Read: if most states move by less than about 0.02, including thin DRGs does not change the conclusions (DQ-16).

WITH nonthin AS (
    SELECT
        state_abbr,
        SUM(tot_discharges) AS tot_discharges,
        SUM(tot_total_payment) AS tot_payment,
        SUM(tot_submitted_charge) / SUM(tot_total_payment) AS markup_ratio,
        SUM(tot_discharges * median_payment) AS expected_payment,
        SUM(tot_total_payment) / SUM(tot_discharges * median_payment) AS payment_index
    FROM (
        SELECT
            f.ccn,
            f.tot_discharges,
            f.tot_total_payment,
            f.tot_submitted_charge,
            b.median_payment,
            h.state_abbr
    FROM workspace.cms.gold_fact_inpatient f
    JOIN workspace.cms.gold_dim_hospital h  ON f.ccn = h.ccn
    JOIN workspace.cms.gold_drg_benchmark b ON f.drg_cd = b.drg_cd
    WHERE h.is_maryland = false
    AND b.n_hospitals >= 30
)
GROUP BY state_abbr
)

SELECT n.state_abbr,
    ROUND(s.payment_index, 3) AS main_index,
    ROUND(n.payment_index, 3) AS nonthin_index,
    ROUND(s.payment_index - n.payment_index, 3) AS diff
FROM nonthin n 
JOIN workspace.cms.gold_state_summary s ON n.state_abbr = s.state_abbr
ORDER BY ABS(diff) DESC;

state_abbr,main_index,nonthin_index,diff
DC,1.383,1.389,-0.006
NY,1.449,1.453,-0.004
MA,1.226,1.229,-0.003
CA,1.427,1.43,-0.002
NM,1.035,1.038,-0.002
OR,1.21,1.212,-0.002
DE,1.085,1.087,-0.002
UT,1.036,1.037,-0.002
HI,1.414,1.416,-0.002
NJ,1.163,1.164,-0.001


In [0]:
--- Payment variation (Week 7 input, resume figure): dollars paid above the state median for each DRG.
--- state_drg: one row per state x DRG, Maryland excluded (DQ-13); payment per discharge = totals over totals (DQ-04).
--- drg_median: plain median of state payment per discharge for each DRG, so each state counts once.
--- Excess = (state payment per discharge - DRG median) x state discharges, summed over above-median states only.
--- Wording (DQ-23): this is payment variation, not savings. Much of it reflects wage-index, teaching
--- and safety-net adjustments built into IPPS, and cells with 10 or fewer discharges are suppressed (DQ-03).

WITH state_drg AS (
    SELECT
        state_abbr,
        drg_cd,
        SUM(tot_discharges) AS tot_discharges,
        SUM(tot_total_payment) AS tot_payment,
        SUM(tot_total_payment) / SUM(tot_discharges) AS payment_per_discharge
    FROM (
        SELECT
            f.ccn,
            f.tot_discharges,
            f.tot_total_payment,
            f.drg_cd,
            h.state_abbr
            
        FROM workspace.cms.gold_fact_inpatient f
        JOIN workspace.cms.gold_dim_hospital h  ON f.ccn = h.ccn
        WHERE h.is_maryland = false
    )
    GROUP BY state_abbr, drg_cd
),
drg_median AS (
    SELECT
        drg_cd,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY payment_per_discharge) AS median_payment
    FROM state_drg
    GROUP BY drg_cd
)

SELECT
    SUM(CASE WHEN state_drg.payment_per_discharge > drg_median.median_payment THEN (state_drg.payment_per_discharge - drg_median.median_payment) * state_drg.tot_discharges ELSE 0 END) AS excess_payment,
    SUM(state_drg.tot_payment) AS total_payment,
    SUM(CASE WHEN state_drg.payment_per_discharge > drg_median.median_payment THEN (state_drg.payment_per_discharge - drg_median.median_payment) * state_drg.tot_discharges ELSE 0 END) / SUM(state_drg.tot_payment) AS excess_share
    
FROM state_drg
JOIN drg_median ON state_drg.drg_cd = drg_median.drg_cd;

excess_payment,total_payment,excess_share
8.567743802057304E9,87830892845.0025098,0.09754818065184682


In [0]:
--- Urban vs rural premium (H4), raw and case-mix adjusted, from gold_ruca_summary.
--- raw_ratio = urban payment per discharge / rural; adjusted_ratio = urban payment index / rural.
--- Expected: about 1.41 (41% more raw) and 1.11 (11% more after case mix). Rounded for display only (DQ-05).
SELECT
    ROUND(MAX(CASE WHEN urban_rural = 'Urban' THEN payment_per_discharge END)
        / MAX(CASE WHEN urban_rural = 'Rural' THEN payment_per_discharge END), 3) AS raw_ratio,
    ROUND(MAX(CASE WHEN urban_rural = 'Urban' THEN payment_index END)
        / MAX(CASE WHEN urban_rural = 'Rural' THEN payment_index END), 3) AS adjusted_ratio
FROM workspace.cms.gold_ruca_summary
WHERE ruca_level = 'urban_rural';

raw_ratio,adjusted_ratio
1.413,1.111
